# Rollout Visualisation — Video + Joint Angles + Torques + Velocities

Downloads the video and matching CSV from a W&B run, then shows:
1. **Video** of the rollout
2. **Manipulator joint angles** (wrapped to [−π, +π])
3. **Finger / gripper positions** (separate graph)
4. **All velocities** overlaid
5. **All torques** overlaid

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, json
from IPython.display import Video, display, HTML

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1 — Configuration

In [ ]:
# ──────────── EDIT THESE ────────────
ENTITY  = "weissma6-zhaw-school-of-engineering"
PROJECT = "UR10_pick_ppo"

# Leave None to auto-select the most recent finished run
RUN_ID  = None
# ────────────────────────────────────


## 2 — Fetch run & files

In [ ]:
api = wandb.Api()

if RUN_ID:
    run = api.run(f'{ENTITY}/{PROJECT}/{RUN_ID}')
else:
    runs = api.runs(f'{ENTITY}/{PROJECT}', filters={'state': 'finished'},
                    order='-created_at', per_page=1)
    run = runs[0]

print(f'Run: {run.name}  ({run.id})')
print(f'State: {run.state}  |  Runtime: {run.summary.get("_runtime", 0)/60:.1f} min')

run_files = list(run.files())
print(f'\nFiles ({len(run_files)}):')
for f in run_files:
    print(f'  {f.name}  ({f.size:,} bytes)')

In [ ]:
os.makedirs('dl', exist_ok=True)

# Find last video
video_files = [f for f in run_files if f.name.endswith('.mp4')]
assert video_files, 'No video found in this run.'
vf = video_files[-1]
vf.download(root='dl', replace=True)
video_path = os.path.join('dl', vf.name)
print(f'Video: {vf.name}')

# Find matching CSV (same base name)
video_base = os.path.splitext(vf.name)[0]
csv_match = [f for f in run_files if f.name == video_base + '.csv']
if not csv_match:
    csv_match = [f for f in run_files if f.name.endswith('.csv') and 'rollout_coords' not in f.name]
assert csv_match, 'No matching CSV found — re-run training with updated UR10_ppo.py.'
cf = csv_match[-1]
cf.download(root='dl', replace=True)
csv_path = os.path.join('dl', cf.name)
print(f'CSV:   {cf.name}')

df = pd.read_csv(csv_path)
print(f'\nLoaded {len(df)} rows × {len(df.columns)} columns')
display(df.head(3))

## 3 — Video

In [ ]:
display(Video(video_path, embed=True, width=700))

## 4 — Detect available data channels

In [ ]:
cols = list(df.columns)

qpos_cols   = sorted([c for c in cols if c.startswith('qpos_')])
vel_cols    = sorted([c for c in cols if c.startswith('vel_')])
torque_cols = sorted([c for c in cols if c.startswith('torque_')])
ctrl_cols   = sorted([c for c in cols if c.startswith('ctrl_')])

print(f'qpos   columns ({len(qpos_cols)}): {qpos_cols}')
print(f'vel    columns ({len(vel_cols)}): {vel_cols}')
print(f'torque columns ({len(torque_cols)}): {torque_cols}')
print(f'ctrl   columns ({len(ctrl_cols)}): {ctrl_cols}')

## 5 — Classify joints: arm vs. fingers

UR10 has 6 arm joints, then finger/gripper joints.  
Edit `N_ARM_JOINTS` below if your model differs.

In [ ]:
# ──────────── EDIT IF NEEDED ────────────
N_ARM_JOINTS = 6   # first 6 qpos entries = arm revolute joints
# ────────────────────────────────────────

arm_qpos_cols    = qpos_cols[:N_ARM_JOINTS]
finger_qpos_cols = qpos_cols[N_ARM_JOINTS:]

print(f'Arm joints ({len(arm_qpos_cols)}):    {arm_qpos_cols}')
print(f'Finger joints ({len(finger_qpos_cols)}): {finger_qpos_cols}')

# Also split velocities and torques the same way if they match
arm_vel_cols    = vel_cols[:N_ARM_JOINTS]
finger_vel_cols = vel_cols[N_ARM_JOINTS:]
arm_torque_cols    = torque_cols[:N_ARM_JOINTS]
finger_torque_cols = torque_cols[N_ARM_JOINTS:]

## 6 — Manipulator Joint Angles [−π, +π]

In [ ]:
if arm_qpos_cols:
    t = df['timestep'].values
    fig, ax = plt.subplots(figsize=(12, 5))

    colors = plt.cm.tab10.colors
    for i, col in enumerate(arm_qpos_cols):
        # Wrap to [-pi, pi]
        angles = np.arctan2(np.sin(df[col].values), np.cos(df[col].values))
        label = col.replace('qpos_', '')
        ax.plot(t, angles, lw=1.3, color=colors[i % len(colors)], label=label)

    ax.set_ylim(-np.pi * 1.1, np.pi * 1.1)
    ax.axhline( np.pi, color='gray', ls=':', alpha=0.4)
    ax.axhline(-np.pi, color='gray', ls=':', alpha=0.4)
    ax.axhline(0,      color='gray', ls=':', alpha=0.2)
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Angle (rad)')
    ax.set_title(f'Arm Joint Angles — {run.name}', fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig('arm_joint_angles.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No qpos columns found — re-run training with updated UR10_ppo.py.')

## 7 — Finger / Gripper Positions

In [ ]:
if finger_qpos_cols:
    t = df['timestep'].values
    fig, ax = plt.subplots(figsize=(12, 4))

    colors = plt.cm.Set2.colors
    for i, col in enumerate(finger_qpos_cols):
        label = col.replace('qpos_', '')
        ax.plot(t, df[col].values, lw=1.3, color=colors[i % len(colors)], label=label)

    ax.set_xlabel('Timestep')
    ax.set_ylabel('Position')
    ax.set_title(f'Finger / Gripper Positions — {run.name}', fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.savefig('finger_positions.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No finger qpos columns found.')
    print('If your robot has no gripper joints, all joints are classified as arm joints.')

## 8 — All Velocities (one graph)

In [ ]:
if vel_cols:
    t = df['timestep'].values
    fig, ax = plt.subplots(figsize=(12, 5))

    colors = plt.cm.tab20.colors
    for i, col in enumerate(vel_cols):
        label = col.replace('vel_', 'v')
        ax.plot(t, df[col].values, lw=1.0, color=colors[i % len(colors)],
                label=label, alpha=0.85)

    ax.axhline(0, color='gray', ls=':', alpha=0.3)
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Velocity (rad/s or m/s)')
    ax.set_title(f'All Velocities — {run.name}', fontweight='bold')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    plt.tight_layout()
    plt.savefig('all_velocities.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No velocity columns found.')

## 9 — All Torques (one graph)

In [ ]:
if torque_cols:
    t = df['timestep'].values
    fig, ax = plt.subplots(figsize=(12, 5))

    colors = plt.cm.tab20.colors
    for i, col in enumerate(torque_cols):
        label = col.replace('torque_', '')
        ax.plot(t, df[col].values, lw=1.0, color=colors[i % len(colors)],
                label=label, alpha=0.85)

    ax.axhline(0, color='gray', ls=':', alpha=0.3)
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Torque (N·m)')
    ax.set_title(f'All Torques — {run.name}', fontweight='bold')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    plt.tight_layout()
    plt.savefig('all_torques.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No torque columns found.')

## 10 — Summary Statistics

In [ ]:
data_groups = [
    ('Joint Angles (arm)',  arm_qpos_cols),
    ('Finger Positions',    finger_qpos_cols),
    ('Velocities',          vel_cols),
    ('Torques',             torque_cols),
]

for title, group_cols in data_groups:
    if group_cols:
        print(f'\n── {title} ──')
        summary = df[group_cols].describe().T
        summary.columns = [c.capitalize() for c in summary.columns]
        display(summary[['Mean', 'Std', 'Min', 'Max']])
